<a href="https://colab.research.google.com/github/bahmedx/RL_CartPole/blob/main/cartpole_dqn_optimized.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# CartPole DQN: Interactive Training & Evaluation

Run these cells sequentially to install dependencies, optimize hyperparameters, train the agent,
**view the training curve plot**, and **play the evaluation video**.

**Goal:** reach a 100-episode moving-average score of >= 475 (the official "solved" threshold
for CartPole-v1) using an agent whose every hyperparameter is discovered by Optuna rather
than hardcoded.

In [ ]:
!pip install gymnasium[classic-control] torch numpy matplotlib imageio imageio-ffmpeg optuna

In [ ]:
import os
import json
import random
import logging
from collections import deque

import gymnasium as gym
import numpy as np
import optuna
import torch
import torch.nn as nn
import torch.optim as optim
import matplotlib.pyplot as plt
import imageio
from IPython.display import Video, display

optuna.logging.set_verbosity(optuna.logging.WARNING)
logging.getLogger("imageio_ffmpeg").setLevel(logging.ERROR)

In [ ]:
ENV_NAME = "CartPole-v1"
SOLVED_SCORE = 475.0          # official CartPole-v1 "solved" bar, not a model hyperparameter
SOLVED_WINDOW = 100            # episodes averaged for the solved check (Gymnasium convention)

# --- Experiment / search budget (infrastructure knobs, not agent hyperparameters) ---
N_TRIALS = 30
TRIAL_TIMEOUT_SEC = 1200
TRIAL_EPISODES = 300
FINAL_MAX_EPISODES = 800
SEED = 42


def set_global_seed(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)


set_global_seed(SEED)

### Model, Replay Buffer, and Agent

In [ ]:
class DynamicQNetwork(nn.Module):
    def __init__(self, state_size: int, action_size: int, hidden_dim: int):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(state_size, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, action_size),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.net(x)

In [ ]:
class ReplayBuffer:
    def __init__(self, capacity: int):
        self.buffer = deque(maxlen=capacity)

    def push(self, state, action, reward, next_state, done):
        self.buffer.append((state, action, reward, next_state, done))

    def sample(self, batch_size: int):
        states, actions, rewards, next_states, dones = zip(*random.sample(self.buffer, batch_size))
        return np.array(states), np.array(actions), np.array(rewards), np.array(next_states), np.array(dones)

    def __len__(self):
        return len(self.buffer)

In [ ]:
class AdaptiveDQNAgent:
    """
    Every knob that affects learning (network width, learning rate and its decay,
    discount factor, target-network Polyak coefficient, batch size, replay buffer
    size, and the epsilon-greedy schedule) lives in `params`, which is produced by
    Optuna's search rather than hardcoded here.
    """

    def __init__(self, state_size: int, action_size: int, params: dict):
        self.state_size = state_size
        self.action_size = action_size
        self.params = params
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

        self.q_network = DynamicQNetwork(state_size, action_size, params["hidden_dim"]).to(self.device)
        self.target_network = DynamicQNetwork(state_size, action_size, params["hidden_dim"]).to(self.device)
        self.target_network.load_state_dict(self.q_network.state_dict())
        self.optimizer = optim.Adam(self.q_network.parameters(), lr=params["learning_rate"])
        self.scheduler = optim.lr_scheduler.ExponentialLR(self.optimizer, gamma=params["lr_decay"])

        self.memory = ReplayBuffer(params["buffer_capacity"])
        self.epsilon = params.get("epsilon_start", 1.0)
        self.optimizer_stepped = False

    def act(self, state: np.ndarray) -> int:
        if np.random.rand() <= self.epsilon:
            return random.randrange(self.action_size)
        state_tensor = torch.FloatTensor(state).unsqueeze(0).to(self.device)
        with torch.no_grad():
            q_values = self.q_network(state_tensor)
        return int(np.argmax(q_values.cpu().data.numpy()))

    def step(self, state, action, reward, next_state, done):
        self.memory.push(state, action, reward, next_state, done)
        if len(self.memory) > self.params["batch_size"]:
            self.learn()

    def learn(self):
        states, actions, rewards, next_states, dones = self.memory.sample(self.params["batch_size"])
        states = torch.FloatTensor(states).to(self.device)
        actions = torch.LongTensor(actions).unsqueeze(1).to(self.device)
        rewards = torch.FloatTensor(rewards).unsqueeze(1).to(self.device)
        next_states = torch.FloatTensor(next_states).to(self.device)
        dones = torch.FloatTensor(dones).unsqueeze(1).to(self.device)

        # Double-DQN target: select the next action with the online network,
        # evaluate it with the target network. Reduces DQN's overestimation bias
        # and materially improves how reliably CartPole reaches the 475 bar.
        with torch.no_grad():
            next_actions = self.q_network(next_states).argmax(1, keepdim=True)
            next_q = self.target_network(next_states).gather(1, next_actions)
            target_q = rewards + (self.params["gamma"] * next_q * (1 - dones))

        current_q = self.q_network(states).gather(1, actions)
        loss = nn.MSELoss()(current_q, target_q)

        self.optimizer.zero_grad()
        loss.backward()
        nn.utils.clip_grad_norm_(self.q_network.parameters(), max_norm=10.0)
        self.optimizer.step()
        self.optimizer_stepped = True

        tau = self.params["tau"]
        for target_param, param in zip(self.target_network.parameters(), self.q_network.parameters()):
            target_param.data.copy_(tau * param.data + (1.0 - tau) * target_param.data)

    def update_epsilon_and_lr(self):
        self.epsilon = max(self.params["epsilon_min"], self.epsilon * self.params["epsilon_decay"])
        if self.optimizer_stepped:
            self.scheduler.step()

In [ ]:
def run_episodes(agent: AdaptiveDQNAgent, env, n_episodes: int, trial: "optuna.Trial | None" = None,
                  report_window: int = 100):
    """Shared training loop used by both the Optuna objective and the final run."""
    scores = []
    for episode in range(1, n_episodes + 1):
        state, _ = env.reset()
        score, done = 0.0, False
        while not done:
            action = agent.act(state)
            next_state, reward, terminated, truncated, _ = env.step(action)
            done = terminated or truncated
            # Bootstrap only on true termination, not on time-limit truncation,
            # so the agent isn't taught that hitting the step cap is "failure".
            agent.step(state, action, reward, next_state, terminated)
            state = next_state
            score += reward

        agent.update_epsilon_and_lr()
        scores.append(score)
        moving_avg = np.mean(scores[-report_window:]) if len(scores) >= report_window else np.mean(scores)

        if trial is not None:
            trial.report(moving_avg, episode)
            if trial.should_prune():
                raise optuna.exceptions.TrialPruned()

        yield episode, score, moving_avg, scores

print("Classes and helper functions loaded successfully!")

### Run Hyperparameter Optimization & Final Training

In [ ]:
def objective(trial: optuna.Trial) -> float:
    params = {
        "learning_rate": trial.suggest_float("learning_rate", 1e-4, 1e-2, log=True),
        "lr_decay": trial.suggest_float("lr_decay", 0.99, 0.9999),
        "gamma": trial.suggest_float("gamma", 0.95, 0.999),
        "tau": trial.suggest_float("tau", 0.001, 0.05),
        "batch_size": trial.suggest_categorical("batch_size", [32, 64, 128]),
        "buffer_capacity": trial.suggest_categorical("buffer_capacity", [5000, 10000, 20000, 50000]),
        "hidden_dim": trial.suggest_categorical("hidden_dim", [64, 128, 256]),
        "epsilon_decay": trial.suggest_float("epsilon_decay", 0.98, 0.999),
        "epsilon_min": trial.suggest_float("epsilon_min", 0.001, 0.05),
    }

    env = gym.make(ENV_NAME)
    state_size = env.observation_space.shape[0]
    action_size = env.action_space.n
    agent = AdaptiveDQNAgent(state_size, action_size, params)

    scores = []
    try:
        for episode, score, moving_avg, scores in run_episodes(agent, env, TRIAL_EPISODES, trial=trial):
            pass
    finally:
        env.close()

    return float(np.mean(scores[-100:]))

In [ ]:
print("Launching Automated Hyperparameter Optimization...")
study = optuna.create_study(direction="maximize", pruner=optuna.pruners.MedianPruner(n_warmup_steps=150))
study.optimize(objective, n_trials=N_TRIALS, timeout=TRIAL_TIMEOUT_SEC)

best_params = dict(study.best_params)  # every field here came from the search, nothing hardcoded

with open("config.json", "w") as f:
    json.dump(best_params, f, indent=4)

print("\nBest Parameters Found:")
print(json.dumps(best_params, indent=4))

In [ ]:
env = gym.make(ENV_NAME)
state_size = env.observation_space.shape[0]
action_size = env.action_space.n
agent = AdaptiveDQNAgent(state_size, action_size, best_params)

scores = []
best_moving_avg = 0.0
solved_at = None

print("\nStarting Final Training Loop with Best Parameters...")
for episode, score, moving_avg, scores in run_episodes(agent, env, FINAL_MAX_EPISODES,
                                                         report_window=SOLVED_WINDOW):
    if episode % 50 == 0:
        print(f"Episode: {episode}/{FINAL_MAX_EPISODES} | Score: {score:.0f} | "
              f"{SOLVED_WINDOW}-Ep Avg: {moving_avg:.2f}")

    if moving_avg > best_moving_avg:
        best_moving_avg = moving_avg
        torch.save(agent.q_network.state_dict(), "cartpole_dqn.pth")

    if moving_avg >= SOLVED_SCORE and episode >= SOLVED_WINDOW:
        solved_at = episode
        print(f"\nEnvironment solved in {episode} episodes! "
              f"{SOLVED_WINDOW}-episode average = {moving_avg:.2f}")
        break

env.close()

if solved_at is None:
    print(f"\nDid not reach the {SOLVED_SCORE} bar within {FINAL_MAX_EPISODES} episodes. "
          f"Best {SOLVED_WINDOW}-episode average achieved: {best_moving_avg:.2f}. "
          "The checkpoint for that best average was still saved to cartpole_dqn.pth.")

### Display Training Curve

In [ ]:
plt.figure(figsize=(10, 5))
plt.plot(scores, label="Raw Score", alpha=0.4, color="blue")
if len(scores) >= SOLVED_WINDOW:
    moving_avg_line = np.convolve(scores, np.ones(SOLVED_WINDOW) / SOLVED_WINDOW, mode="valid")
    padded_avg = np.concatenate((np.full(SOLVED_WINDOW - 1, np.nan), moving_avg_line))
    plt.plot(padded_avg, label=f"{SOLVED_WINDOW}-Episode Moving Avg", color="red", linewidth=2)
plt.axhline(SOLVED_SCORE, color="green", linestyle="--", alpha=0.6, label=f"Solved ({SOLVED_SCORE:.0f})")
plt.title("Final Optimized DQN Performance")
plt.ylabel("Score (Steps Balanced)")
plt.xlabel("Episode")
plt.legend()
plt.grid(True, linestyle="--", alpha=0.5)
plt.savefig("training_curve.png", dpi=150, bbox_inches="tight")
plt.show()

### Render and Display Evaluation Demo Video

In [ ]:
print("Running evaluation simulation and recording video...")
eval_env = gym.make(ENV_NAME, render_mode="rgb_array")
q_network = DynamicQNetwork(state_size, action_size, best_params["hidden_dim"]).to(agent.device)
q_network.load_state_dict(torch.load("cartpole_dqn.pth", map_location=agent.device, weights_only=True))
q_network.eval()

state, _ = eval_env.reset(seed=SEED)
done = False
frames = []
step_count = 0

while not done:
    frame = eval_env.render()
    padded_frame = np.pad(frame, ((0, 0), (4, 4), (0, 0)), mode="edge")
    frames.append(padded_frame)

    state_tensor = torch.FloatTensor(state).unsqueeze(0).to(agent.device)
    with torch.no_grad():
        q_values = q_network(state_tensor)

    action = int(torch.argmax(q_values).item())
    state, reward, terminated, truncated, _ = eval_env.step(action)
    done = terminated or truncated
    step_count += 1

eval_env.close()
print(f"Evaluation complete! Balanced for {step_count} steps.")

video_path = "cartpole_demo.mp4"
imageio.mimsave(video_path, frames, fps=30)
display(Video(video_path, embed=True))